In [1]:
# --- Cell 1: Load the dataset and confirm its structure ---

# Bring in the tools we need: dotenv to load the token, os is its helper, and
# load_dataset to pull the bird dataset from Hugging Face.
from dotenv import load_dotenv
import os
from datasets import load_dataset

# Load the Hugging Face token from the .env file (same secure setup as our script).
# The dataset loader picks this token up automatically for higher rate limits.
load_dotenv()

# Load ALL splits of the bird dataset (train, validation, and test if they exist).
# Leaving out split="train" gives us the whole thing instead of just one piece.
dataset = load_dataset("yashikota/birds-525-species-image-classification")

# Print the overall structure: the split names, their features, and how many
# images are in each split.
print(dataset)


c:\workspace\neumont\AvianVision\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 84635
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 2625
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 2625
    })
})


In [2]:
# --- Cell 2: Class distribution in the training split ---

import pandas as pd

# Grab the label info: the species-name lookup, and the label number of every training image.
label_feature = dataset["train"].features["label"]
label_numbers = dataset["train"]["label"]

# Count how many images fall under each label number, kept in label order.
counts = pd.Series(label_numbers).value_counts().sort_index()

# Swap the label numbers for their species names so the output is readable.
counts.index = [label_feature.int2str(i) for i in counts.index]

# Print summary stats describing how balanced (or not) the classes are.
print(f"Number of classes:       {len(counts)}")
print(f"Total training images:   {counts.sum()}")
print(f"Fewest images in a class:{counts.min()}")
print(f"Most images in a class:  {counts.max()}")
print(f"Mean images per class:   {counts.mean():.1f}")
print(f"Median images per class: {counts.median():.1f}")
print()

# Show the 10 species with the FEWEST images — the likely weak spots for the model.
print("10 species with the FEWEST images:")
print(counts.sort_values().head(10))
print()

# Show the 10 species with the MOST images.
print("10 species with the MOST images:")
print(counts.sort_values(ascending=False).head(10))


Number of classes:       525
Total training images:   84635
Fewest images in a class:130
Most images in a class:  263
Mean images per class:   161.2
Median images per class: 158.0

10 species with the FEWEST images:
AMETHYST WOODSTAR          130
RED TAILED THRUSH          130
SNOWY PLOVER               130
PATAGONIAN SIERRA FINCH    130
NORTHERN RED BISHOP        130
EURASIAN GOLDEN ORIOLE     130
BLACK FRANCOLIN            131
MASKED LAPWING             131
INDIAN BUSTARD             131
GO AWAY BIRD               131
Name: count, dtype: int64

10 species with the MOST images:
RUFOUS TREPE             263
HOUSE FINCH              248
OVENBIRD                 233
D-ARNAUDS BARBET         233
ASIAN GREEN BEE EATER    220
SWINHOES PHEASANT        216
WOOD DUCK                214
CASPIAN TERN             213
RED BILLED TROPICBIRD    212
WOOD THRUSH              211
Name: count, dtype: int64


In [3]:
# --- Cell 3: Find the mystery label (defined but with no images) ---

from collections import Counter

# All the class names the dataset defines (we saw this was 526).
all_label_names = label_feature.names
print(f"Total class names defined: {len(all_label_names)}")

# Count images per label number across EVERY split (train + validation + test).
total_counts = Counter()
for split_name in dataset:
    total_counts.update(dataset[split_name]["label"])

# Convert the label numbers that DID appear into their species names.
present = {label_feature.int2str(i) for i in total_counts}

# Any defined name that never showed up in an image is our phantom class.
missing = [name for name in all_label_names if name not in present]

print(f"Class names that appear in images: {len(present)}")
print(f"Class names with NO images in ANY split: {missing}")



Total class names defined: 526
Class names that appear in images: 526
Class names with NO images in ANY split: []


In [4]:
# --- Cell 4: Find the class missing from the TRAINING split ---

# The species names that DO appear in training (from Cell 2's counts).
train_present = set(counts.index)

# Any defined class name that never shows up in the training split.
missing_from_train = [name for name in all_label_names if name not in train_present]
print(f"Classes present in train: {len(train_present)}")
print(f"Classes MISSING from train: {missing_from_train}")

# For each missing class, show how many images it has in every split.
for name in missing_from_train:
    label_id = label_feature.str2int(name)
    print(f"\nImage counts for '{name}' (label {label_id}):")
    for split_name in dataset:
        n = sum(1 for lbl in dataset[split_name]["label"] if lbl == label_id)
        print(f"  {split_name}: {n}")


Classes present in train: 525
Classes MISSING from train: ['PARAKETT AUKLET']

Image counts for 'PARAKETT AUKLET' (label 381):
  train: 0
  validation: 5
  test: 0


In [5]:
# --- Cell 5: Confirm the two Parakeet Auklet labels are duplicates ---

# Find every label name that looks like the Parakeet Auklet.
auklet_labels = [(i, name) for i, name in enumerate(all_label_names)
                 if "PARAKETT" in name or "AUKLET" in name]

# Show each one's EXACT spelling (repr reveals hidden double spaces) and its
# image counts per split, so we can see they're the same bird split in two.
for label_id, name in auklet_labels:
    print(f"label {label_id}: {name!r}  (text length {len(name)})")
    for split_name in dataset:
        n = sum(1 for lbl in dataset[split_name]["label"] if lbl == label_id)
        print(f"    {split_name}: {n}")
    print()

label 168: 'CRESTED AUKLET'  (text length 14)
    train: 160
    validation: 5
    test: 5

label 380: 'PARAKETT  AUKLET'  (text length 16)
    train: 155
    validation: 0
    test: 5

label 381: 'PARAKETT AUKLET'  (text length 15)
    train: 0
    validation: 5
    test: 0



In [6]:
# --- Cell 6: Scan every image to make sure it opens (corruption check) ---

# We'll record any image that fails to open, along with where it is and the error.
bad_images = []

# Go through each split and try to fully load every image.
for split_name in dataset:
    split = dataset[split_name]
    print(f"Scanning {split_name} ({len(split)} images)...")
    for i, example in enumerate(split):
        try:
            # Force the image to fully decode; a corrupt file raises an error here.
            example["image"].load()
        except Exception as e:
            # Save the split, position, and error message for anything that fails.
            bad_images.append((split_name, i, str(e)))
        # Print a heartbeat every 10,000 images so we know it's still working.
        if (i + 1) % 10000 == 0:
            print(f"  ...checked {i + 1} images")

# Report the results.
print(f"\nScan complete. Corrupted or unreadable images: {len(bad_images)}")
for item in bad_images[:20]:
    print(item)

Scanning train (84635 images)...
  ...checked 10000 images
  ...checked 20000 images
  ...checked 30000 images
  ...checked 40000 images
  ...checked 50000 images
  ...checked 60000 images
  ...checked 70000 images
  ...checked 80000 images
Scanning validation (2625 images)...
Scanning test (2625 images)...

Scan complete. Corrupted or unreadable images: 0


In [7]:
# --- Cell 7: Check image color modes and dimensions ---

from collections import Counter

# Tally the color mode (e.g. RGB, grayscale) and the size of every image.
mode_counts = Counter()
size_counts = Counter()

# Look at every image across all splits.
for split_name in dataset:
    split = dataset[split_name]
    for example in split:
        img = example["image"]
        mode_counts[img.mode] += 1   # color mode, e.g. "RGB" or "L" (grayscale)
        size_counts[img.size] += 1   # (width, height) in pixels

# Show which color modes appear and how many images use each.
print("Color modes:")
for mode, n in mode_counts.most_common():
    print(f"  {mode}: {n}")

# Show the most common image sizes.
print("\nMost common image sizes:")
for size, n in size_counts.most_common(10):
    print(f"  {size}: {n}")

print(f"\nNumber of distinct sizes: {len(size_counts)}")


Color modes:
  RGB: 89885

Most common image sizes:
  (224, 224): 89674
  (170, 196): 1
  (4763, 3421): 1
  (3100, 2810): 1
  (2670, 2186): 1
  (2086, 3162): 1
  (1463, 2826): 1
  (1756, 1456): 1
  (1829, 1171): 1
  (1543, 928): 1

Number of distinct sizes: 212


## Dataset Integrity Summary

**Dataset:** `yashikota/birds-525-species-image-classification`

### Findings
- **Splits:** train 84,635 · validation 2,625 · test 2,625 (89,885 total) — already pre-split.
- **Classes:** 526 labels defined, but **525 true species**. The extra label is a whitespace-typo duplicate of the Parakeet Auklet:
  - label 380 `'PARAKETT  AUKLET'` (two spaces) → 155 / 0 / 5
  - label 381 `'PARAKETT AUKLET'` (one space) → 0 / 5 / 0
  - Combined = 155 / 5 / 5, one normal class.
- **Class balance:** mild (~2:1) — 130–263 images/class, mean 161. No weighting/oversampling needed.
- **Corrupted images:** 0 across all 89,885.
- **Color mode:** 100% RGB.
- **Image size:** 89,674 already 224×224; 211 outliers of varying sizes.

### Preprocessing implications
1. Resize all images to 224×224 (handles the 211 outliers).
2. Normalize pixels using EfficientNetB0's expected preprocessing.
3. Merge Parakeet Auklet labels 380 + 381 into one class.
4. No color conversion, corruption filtering, or class rebalancing needed.
